# Códigos Exemplo no Apache Spark

Para utilizar o Apache Spark é necessário utilizar **Java 8** ou **Java 11**. 

Você pode definir a versão do Java no terminal através dos comandos:

```bash
export JAVA_HOME="/Library/Java/JavaVirtualMachines/jdk-11.jdk/Contents/Home"
export PATH=$JAVA_HOME/bin:$PATH
```

A versão Scala esperada é 2.12 ou 2.13

Para facilitar a visualização, vamos utilizar durante a aula este notebook, porém para utilizar na sua IDE de preferência, podemos adicionar no build.sbt:

```
name := "AulaSparkDemo"

version := "0.1"

scalaVersion := "2.12.18"

libraryDependencies ++= Seq(
  "org.apache.spark" %% "spark-sql" % "3.5.0",
  "org.slf4j" % "slf4j-simple" % "2.0.9"
)
```

Também é possível utilizar o Maven, atavés do pom.xml. Verifique o pom.xml exemplo no site oficial do Spark: https://spark.apache.org
Sendo que a versão mais recente do Spark foi lançada recentemente, iremos utilizar a 3.5. Verifiquem as principais diferenças para a 4.2 no site, a implementação e as APIs continuam majoritariamente as mesmas. 

In [1]:
// Importando o Apache Spark no Almond
import $ivy.`org.apache.spark::spark-sql:3.5.0`

import org.apache.spark.sql.SparkSession

// Inicializa o Spark em modo Local utilizando 2 threads para simular os Workers
val spark = SparkSession.builder()
  .appName("AulaSparkScalaDemo")
  .master("local[2]")
  .getOrCreate()

import spark.implicits._
spark.sparkContext.setLogLevel("ERROR")

val sc = spark.sparkContext

println(s"Driver App Name: ${sc.appName}")
println(s"Modo de Execução (Master): ${sc.master}")
println(s"Spark UI disponível em: ${sc.uiWebUrl.getOrElse("http://localhost:4040")}")

Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar.sha1
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/30 19:38:31 WARN Utils: Your hostname, MacBook-Pro-de-Larissa.local resolves to a loopback address: 127.0.0.1; using 191.52.96.121 instead (on interface en0)
26/10/30 19:38:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/10/30 19:38:32 INFO SparkContext: Running Spark version 3.5.0
26/10/30 19:38:32 INFO SparkContext: OS info Mac OS X, 15.7.3, aarch64
26/10/30 19:38:

Driver App Name: AulaSparkScalaDemo
Modo de Execução (Master): local[2]
Spark UI disponível em: http://191.52.96.121:4041


import $ivy.$
import org.apache.spark.sql.SparkSession
spark: SparkSession = org.apache.spark.sql.SparkSession@698c9913
import spark.implicits._
sc: org.apache.spark.SparkContext = org.apache.spark.SparkContext@62b95b7a

In [2]:
// Dados brutos simulando um arquivo de vendas em formato CSV (String)
val dadosBrutos = Seq(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)

// Paralelizando a coleção para criar um RDD[String]
val rddLinhas = sc.parallelize(dadosBrutos)


dadosBrutos: Seq[String] = List(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)
rddLinhas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[0] at parallelize at cmd2.sc:12

In [3]:
// A. Verificar a quantidade padrão de partições criada pelo Spark
println(s"\n[Partições] Quantidade inicial de partições: ${rddLinhas.getNumPartitions}")

// B. Ver exatamente QUAIS DADOS foram parar em QUAL PARTIÇÃO (usando o índice)
println("\n--- Distribuição dos Dados por Partição ---")
rddLinhas.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)



[Partições] Quantidade inicial de partições: 2

--- Distribuição dos Dados por Partição ---
Partição [0] -> Conteúdo: '1,Ana,Eletrônicos,2500.00,SP'
Partição [0] -> Conteúdo: '2,Bruno,Roupas,150.50,RJ'
Partição [0] -> Conteúdo: '3,Carla,Eletrônicos,3200.00,SP'
Partição [1] -> Conteúdo: '4,Diego,Higiene,45.00,MG'
Partição [1] -> Conteúdo: '5,Elena,Roupas,220.00,SP'
Partição [1] -> Conteúdo: '6,Fernando,Eletrônicos,1800.00,RJ'


In [4]:
// C. Forçar o aumento de partições com Shuffle (repartition)
val rddComMaisParticoes = rddLinhas.repartition(3)
println(s"\n[Partições] Quantidade após repartition(3): ${rddComMaisParticoes.getNumPartitions}")

rddComMaisParticoes.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)


[Partições] Quantidade após repartition(3): 3
Partição [0] -> Conteúdo: '1,Ana,Eletrônicos,2500.00,SP'
Partição [0] -> Conteúdo: '6,Fernando,Eletrônicos,1800.00,RJ'
Partição [1] -> Conteúdo: '2,Bruno,Roupas,150.50,RJ'
Partição [1] -> Conteúdo: '4,Diego,Higiene,45.00,MG'
Partição [2] -> Conteúdo: '3,Carla,Eletrônicos,3200.00,SP'
Partição [2] -> Conteúdo: '5,Elena,Roupas,220.00,SP'


rddComMaisParticoes: org.apache.spark.rdd.RDD[String] = MapPartitionsRDD[5] at repartition at cmd4.sc:2

In [5]:
val rddComMenosParticoes = rddComMaisParticoes.coalesce(1)
println(s"[Partições] Quantidade após coalesce(1): ${rddComMenosParticoes.getNumPartitions}")

rddComMenosParticoes.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)


[Partições] Quantidade após coalesce(1): 1
Partição [0] -> Conteúdo: '1,Ana,Eletrônicos,2500.00,SP'
Partição [0] -> Conteúdo: '6,Fernando,Eletrônicos,1800.00,RJ'
Partição [0] -> Conteúdo: '2,Bruno,Roupas,150.50,RJ'
Partição [0] -> Conteúdo: '4,Diego,Higiene,45.00,MG'
Partição [0] -> Conteúdo: '3,Carla,Eletrônicos,3200.00,SP'
Partição [0] -> Conteúdo: '5,Elena,Roupas,220.00,SP'


rddComMenosParticoes: org.apache.spark.rdd.RDD[String] = CoalescedRDD[7] at coalesce at cmd5.sc:1

In [7]:
// Transformação 1: Mapear cada linha CSV em uma tupla estruturada
// (id: Long, cliente: String, categoria: String, valor: Double, estado: String)
val rddTransacoes = rddLinhas.map(linha => {
  val c = linha.split(",")
  (c(0).toLong, c(1), c(2), c(3).toDouble, c(4))
})

rddTransacoes.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição Transacoes [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)

// Transformação 2: Filtrar apenas as vendas do estado de "SP"
val rddSP = rddTransacoes.filter(tupla => tupla._5 == "SP")

rddSP.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição SP [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)

// Transformação 3: Criar um Pair RDD (Chave-Valor) no formato (Categoria, Valor)
val rddCategoriaValor = rddSP.map(tupla => (tupla._3, tupla._4))

rddCategoriaValor.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição Categoria Valor [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)


Partição Transacoes [0] -> Conteúdo: '(1,Ana,Eletrônicos,2500.0,SP)'
Partição Transacoes [0] -> Conteúdo: '(2,Bruno,Roupas,150.5,RJ)'
Partição Transacoes [0] -> Conteúdo: '(3,Carla,Eletrônicos,3200.0,SP)'
Partição Transacoes [1] -> Conteúdo: '(4,Diego,Higiene,45.0,MG)'
Partição Transacoes [1] -> Conteúdo: '(5,Elena,Roupas,220.0,SP)'
Partição Transacoes [1] -> Conteúdo: '(6,Fernando,Eletrônicos,1800.0,RJ)'
Partição SP [0] -> Conteúdo: '(1,Ana,Eletrônicos,2500.0,SP)'
Partição SP [0] -> Conteúdo: '(3,Carla,Eletrônicos,3200.0,SP)'
Partição SP [1] -> Conteúdo: '(5,Elena,Roupas,220.0,SP)'
Partição Categoria Valor [0] -> Conteúdo: '(Eletrônicos,2500.0)'
Partição Categoria Valor [0] -> Conteúdo: '(Eletrônicos,3200.0)'
Partição Categoria Valor [1] -> Conteúdo: '(Roupas,220.0)'


rddTransacoes: org.apache.spark.rdd.RDD[(Long, String, String, Double, String)] = MapPartitionsRDD[14] at map at cmd7.sc:3
rddSP: org.apache.spark.rdd.RDD[(Long, String, String, Double, String)] = MapPartitionsRDD[16] at filter at cmd7.sc:13
rddCategoriaValor: org.apache.spark.rdd.RDD[(String, Double)] = MapPartitionsRDD[18] at map at cmd7.sc:20

In [8]:
// Transformação 4: Somar o valor total por Categoria usando reduceByKey
val rddTotalPorCategoria = rddCategoriaValor.reduceByKey((v1, v2) => v1 + v2)

rddTotalPorCategoria.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(linha => s"Partição [$indexParticao] -> Conteúdo: '$linha'")
}.collect().foreach(println)

Partição [0] -> Conteúdo: '(Roupas,220.0)'
Partição [0] -> Conteúdo: '(Eletrônicos,5700.0)'


rddTotalPorCategoria: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[20] at reduceByKey at cmd8.sc:2

In [9]:
val rddDagCompleta = rddLinhas.map(linha => {
  val c = linha.split(",")
  (c(0).toLong, c(1), c(2), c(3).toDouble, c(4))
}).filter(tupla => tupla._5 == "SP")
.map(tupla => (tupla._3, tupla._4))
.reduceByKey((v1, v2) => v1 + v2)

rddDagCompleta: org.apache.spark.rdd.RDD[(String, Double)] = ShuffledRDD[25] at reduceByKey at cmd9.sc:6

In [10]:
// Visualizacao da DAG
// Exibindo o DAG e a linhagem do RDD diretamente no output do Scala / Console
println("\n=== DAG / Lineage Graph do rddTotalPorCategoria ===")
println(rddTotalPorCategoria.toDebugString)

println("\n=== DAG / Lineage Graph do rddDagCompleta ===")
println(rddDagCompleta.toDebugString)


=== DAG / Lineage Graph do rddTotalPorCategoria ===
(2) ShuffledRDD[20] at reduceByKey at cmd8.sc:2 []
 +-(2) MapPartitionsRDD[18] at map at cmd7.sc:20 []
    |  MapPartitionsRDD[16] at filter at cmd7.sc:13 []
    |  MapPartitionsRDD[14] at map at cmd7.sc:3 []
    |  ParallelCollectionRDD[0] at parallelize at cmd2.sc:12 []

=== DAG / Lineage Graph do rddDagCompleta ===
(2) ShuffledRDD[25] at reduceByKey at cmd9.sc:6 []
 +-(2) MapPartitionsRDD[24] at map at cmd9.sc:5 []
    |  MapPartitionsRDD[23] at filter at cmd9.sc:4 []
    |  MapPartitionsRDD[22] at map at cmd9.sc:1 []
    |  ParallelCollectionRDD[0] at parallelize at cmd2.sc:12 []


In [11]:
// AÇÃO: O Spark executa o plano de processamento (DAG) e retorna o resultado
println("--- Resultado da soma por Categoria em SP (via RDD) ---")
rddTotalPorCategoria.collect().foreach(println)

//println("\n[Atenção] Processo pausado por 3 minutos.")
Thread.sleep(180000)

--- Resultado da soma por Categoria em SP (via RDD) ---
(Roupas,220.0)
(Eletrônicos,5700.0)


## Definindo uma função de particionamento

In [12]:
import org.apache.spark.Partitioner

// 1. Definição do Particionador Customizado
class EstadoPartitioner(val numParticoes: Int) extends Partitioner {
  
  // Retorna o número total de partições que o RDD terá
  override def numPartitions: Int = numParticoes

  // Define a lógica que recebe a CHAVE do RDD e retorna o ÍNDICE da partição (0 a numParticoes - 1)
  override def getPartition(key: Any): Int = {
    key.toString match {
      case "SP" => 0 // Vendas de SP vão obrigatoriamente para a Partição 0
      case "RJ" => 1 // Vendas do RJ vão para a Partição 1
      case _    => 2 // Todos os outros estados (ex: MG) vão para a Partição 2
    }
  }

  // Método opcional para comparação de igualdade de instâncias (usado internamente pelo Spark)
  override def equals(other: Any): Boolean = other match {
    case p: EstadoPartitioner => p.numPartitions == numPartitions
    case _                    => false
  }
}

import org.apache.spark.Partitioner
defined class EstadoPartitioner

In [13]:
// Dados brutos
val dadosVendas = Seq(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)

val rddLinhas = sc.parallelize(dadosVendas)

// Transformar em Pair RDD no formato (Chave = Estado, Valor = LinhaCompleta)
val rddPairPorEstado = rddLinhas.map(linha => {
  val estado = linha.split(",")(4)
  (estado, linha)
})

// Aplicar o Particionador Customizado com 3 partições
val rddCustomizado = rddPairPorEstado.partitionBy(new EstadoPartitioner(3))

// Inspecionar o resultado para provar a alocação aos alunos
println("\n=== Resultado do Particionamento Customizado ===")
rddCustomizado.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(tupla => s"Partição [$indexParticao] -> Estado: ${tupla._1} | Dados: ${tupla._2}")
}.collect().foreach(println)


=== Resultado do Particionamento Customizado ===
Partição [0] -> Estado: SP | Dados: 1,Ana,Eletrônicos,2500.00,SP
Partição [0] -> Estado: SP | Dados: 3,Carla,Eletrônicos,3200.00,SP
Partição [0] -> Estado: SP | Dados: 5,Elena,Roupas,220.00,SP
Partição [1] -> Estado: RJ | Dados: 2,Bruno,Roupas,150.50,RJ
Partição [1] -> Estado: RJ | Dados: 6,Fernando,Eletrônicos,1800.00,RJ
Partição [2] -> Estado: MG | Dados: 4,Diego,Higiene,45.00,MG


dadosVendas: Seq[String] = List(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)
rddLinhas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[26] at parallelize at cmd13.sc:11
rddPairPorEstado: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[27] at map at cmd13.sc:14
rddCustomizado: org.apache.spark.rdd.RDD[(String, String)] = ShuffledRDD[28] at partitionBy at cmd13.sc:20

In [14]:
//Utilizando partitioners comuns no Spark

import org.apache.spark.HashPartitioner

// 1. Coleção de dados base
val dadosVendas = Seq(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)

val rddLinhas = sc.parallelize(dadosVendas)

// 2. Transformar em Pair RDD (Chave = Estado, Valor = LinhaCompleta)
val rddPairPorEstado = rddLinhas.map(linha => {
  val estado = linha.split(",")(4)
  (estado, linha)
})

// 3. Aplicar o HashPartitioner definindo 3 partições desejadas
val rddHash = rddPairPorEstado.partitionBy(new HashPartitioner(3))

// 4. Inspecionar o resultado para os alunos
println("\n=== Resultado do Particionamento com HashPartitioner ===")
rddHash.mapPartitionsWithIndex { (indexParticao, iterador) =>
  iterador.map(tupla => s"Partição [$indexParticao] -> Chave (Estado): ${tupla._1} | Hash: ${tupla._1.hashCode()} | Dados: ${tupla._2}")
}.collect().foreach(println)


=== Resultado do Particionamento com HashPartitioner ===
Partição [0] -> Chave (Estado): RJ | Hash: 2616 | Dados: 2,Bruno,Roupas,150.50,RJ
Partição [0] -> Chave (Estado): RJ | Hash: 2616 | Dados: 6,Fernando,Eletrônicos,1800.00,RJ
Partição [1] -> Chave (Estado): SP | Hash: 2653 | Dados: 1,Ana,Eletrônicos,2500.00,SP
Partição [1] -> Chave (Estado): SP | Hash: 2653 | Dados: 3,Carla,Eletrônicos,3200.00,SP
Partição [1] -> Chave (Estado): MG | Hash: 2458 | Dados: 4,Diego,Higiene,45.00,MG
Partição [1] -> Chave (Estado): SP | Hash: 2653 | Dados: 5,Elena,Roupas,220.00,SP


import org.apache.spark.HashPartitioner
dadosVendas: Seq[String] = List(
  "1,Ana,Eletrônicos,2500.00,SP",
  "2,Bruno,Roupas,150.50,RJ",
  "3,Carla,Eletrônicos,3200.00,SP",
  "4,Diego,Higiene,45.00,MG",
  "5,Elena,Roupas,220.00,SP",
  "6,Fernando,Eletrônicos,1800.00,RJ"
)
rddLinhas: org.apache.spark.rdd.RDD[String] = ParallelCollectionRDD[30] at parallelize at cmd14.sc:15
rddPairPorEstado: org.apache.spark.rdd.RDD[(String, String)] = MapPartitionsRDD[31] at map at cmd14.sc:18
rddHash: org.apache.spark.rdd.RDD[(String, String)] = ShuffledRDD[32] at partitionBy at cmd14.sc:24

In [16]:
import org.apache.spark.HashPartitioner

// 1. Criando um conjunto de dados desproporcional (Data Skew)
// 900 transações de "SP" contra apenas 50 de "RJ" e 50 de "MG"
val dadosComSkew = (1 to 900).map(i => s"$i,Cliente_$i,Eletrônicos,100.00,SP") ++
                   (901 to 950).map(i => s"$i,Cliente_$i,Roupas,50.00,RJ") ++
                   (951 to 1000).map(i => s"$i,Cliente_$i,Higiene,20.00,MG")

val rddInicial = sc.parallelize(dadosComSkew, numSlices = 3)

// 2. Preparando o Pair RDD (Chave = Estado, Valor = ValorDaVenda)
val rddPair = rddInicial.map(linha => {
  val cols = linha.split(",")
  (cols(4), cols(3).toDouble) // (Estado, Valor)
})

// =========================================================================
// ETAPA 1: Aplicando HashPartitioner e evidenciando o Skew
// =========================================================================
val rddParticionado = rddPair.partitionBy(new HashPartitioner(3))

println("\n=== DISTRIBUIÇÃO DAS PARTIÇÕES (Efeito Data Skew) ===")
rddParticionado.mapPartitionsWithIndex { (index, iter) =>
  val registros = iter.toList
  val resumoEstados = registros.groupBy(_._1).mapValues(_.size).mkString(", ")
  Iterator(s"Partição [$index] -> Total Registros: ${registros.size} | Detalhes: [$resumoEstados]")
}.collect().foreach(println)


// =========================================================================
// ETAPA 2: Ação com AGREGADOR (Causa SHUFFLE entre as partições)
// =========================================================================
// O reduceByKey força a movimentação e combinação de dados através da rede/memória
val rddTotalPorEstado = rddParticionado.reduceByKey(_ + _)

println("\n=== RESULTADO FINAL DA AGREGAÇÃO (Pós-Shuffle) ===")
rddTotalPorEstado.collect().foreach(println)

// Pausa para inspeção gráfica do DAG e dos Stages com Shuffle na Spark UI
println(s"\n>>> Abra http://127.0.0.1:4041 para ver o Stage de Shuffle gerado pelo reduceByKey!")
Thread.sleep(60000)

cmd16.sc:25: method mapValues in trait MapOps is deprecated (since 2.13.0): Use .view.mapValues(f). A future version will include a strict version of this method (for now, .view.mapValues(f).toMap).
  val resumoEstados = registros.groupBy(_._1).mapValues(_.size).mkString(", ")
                                              ^



=== DISTRIBUIÇÃO DAS PARTIÇÕES (Efeito Data Skew) ===
Partição [0] -> Total Registros: 50 | Detalhes: [RJ -> 50]
Partição [1] -> Total Registros: 950 | Detalhes: [SP -> 900, MG -> 50]
Partição [2] -> Total Registros: 0 | Detalhes: []

=== RESULTADO FINAL DA AGREGAÇÃO (Pós-Shuffle) ===
(RJ,2500.0)
(SP,90000.0)
(MG,1000.0)

>>> Abra http://127.0.0.1:4040 para ver o Stage de Shuffle gerado pelo reduceByKey!


import org.apache.spark.HashPartitioner
dadosComSkew: IndexedSeq[String] = Vector(
  "1,Cliente_1,Eletrônicos,100.00,SP",
  "2,Cliente_2,Eletrônicos,100.00,SP",
  "3,Cliente_3,Eletrônicos,100.00,SP",
  "4,Cliente_4,Eletrônicos,100.00,SP",
  "5,Cliente_5,Eletrônicos,100.00,SP",
  "6,Cliente_6,Eletrônicos,100.00,SP",
  "7,Cliente_7,Eletrônicos,100.00,SP",
  "8,Cliente_8,Eletrônicos,100.00,SP",
  "9,Cliente_9,Eletrônicos,100.00,SP",
  "10,Cliente_10,Eletrônicos,100.00,SP",
  "11,Cliente_11,Eletrônicos,100.00,SP",
  "12,Cliente_12,Eletrônicos,100.00,SP",
  "13,Cliente_13,Eletrônicos,100.00,SP",
  "14,Cliente_14,Eletrônicos,100.00,SP",
  "15,Cliente_15,Eletrônicos,100.00,SP",
  "16,Cliente_16,Eletrônicos,100.00,SP",
  "17,Cliente_17,Eletrônicos,100.00,SP",
  "18,Cliente_18,Eletrônicos,100.00,SP",
  "19,Cliente_19,Eletrônicos,100.00,SP",
  "20,Cliente_20,Eletrônicos,100.00,SP",
  "21,Cliente_21,Eletrônicos,100.00,SP",
  "22,Cliente_22,Eletrônicos,100.00,SP",
  "23,Cliente_23,Eletrônicos,100.

# SparkSQL

- Leitura de arquivos e criação de Dataframes
  
Reutilizaremos a variável SparkSession criado lá no bloco inicial de código deste notebook. 

Função para ler Dataframes - https://spark.apache.org/docs/latest/api/scala/org/apache/spark/sql/DataFrameReader.html 

In [15]:
import org.apache.spark.sql.{DataFrame, Dataset, Row}

// Exemplo de leitura de CSV
 val dfVendas = spark.read
   .option("header", "true")
   .option("inferSchema", "true")
   .csv("vendas.csv")

dfVendas.printSchema()
dfVendas.show()

root
 |-- id_venda: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_venda: date (nullable = true)

+--------+--------------+----------------+-------+------+----------+
|id_venda|       cliente|       categoria|  valor|estado|data_venda|
+--------+--------------+----------------+-------+------+----------+
|       1| Gabriela Lima|          Livros|3634.52|    MG|2024-03-14|
|       2| Isabela Pires|       Alimentos| 1245.1|    SP|2024-11-02|
|       3|     Ana Silva|     Eletrônicos| 4812.9|    PR|2024-07-22|
|       4|  Hugo Martins|          Roupas| 890.15|    RJ|2024-01-10|
|       5|    Carla Dias|Casa e Decoração|  210.4|    SC|2024-09-05|
|       6|   Bruno Souza|         Higiene|   45.9|    BA|2024-05-18|
|       7|   Diego Alves|     Eletrônicos| 2890.0|    SP|2024-02-11|
|       8|   Elena Costa|          Roupas|  120.5|    MG|2024-08-30|

import org.apache.spark.sql.{DataFrame, Dataset, Row}
dfVendas: DataFrame = [id_venda: int, cliente: string ... 4 more fields]

In [16]:
// Criando o DataFrame diretamente - similar ao que fizemos com o RDD
val dfVendasCriado = Seq(
  (1L, "Ana", "Eletrônicos", 2500.00, "SP"),
  (2L, "Bruno", "Roupas", 150.50, "RJ"),
  (3L, "Carla", "Eletrônicos", 3200.00, "SP"),
  (4L, "Diego", "Higiene", 45.00, "MG"),
  (5L, "Elena", "Roupas", 220.00, "SP"),
  (6L, "Fernando", "Eletrônicos", 1800.00, "RJ")
).toDF("id", "cliente", "categoria", "valor", "estado")

dfVendasCriado.show(5)
dfVendas.printSchema()

+---+-------+-----------+------+------+
| id|cliente|  categoria| valor|estado|
+---+-------+-----------+------+------+
|  1|    Ana|Eletrônicos|2500.0|    SP|
|  2|  Bruno|     Roupas| 150.5|    RJ|
|  3|  Carla|Eletrônicos|3200.0|    SP|
|  4|  Diego|    Higiene|  45.0|    MG|
|  5|  Elena|     Roupas| 220.0|    SP|
+---+-------+-----------+------+------+
only showing top 5 rows

root
 |-- id_venda: integer (nullable = true)
 |-- cliente: string (nullable = true)
 |-- categoria: string (nullable = true)
 |-- valor: double (nullable = true)
 |-- estado: string (nullable = true)
 |-- data_venda: date (nullable = true)



dfVendasCriado: DataFrame = [id: bigint, cliente: string ... 3 more fields]

In [17]:
println(s"df é um Dataset[Row]? ${dfVendas.isInstanceOf[Dataset[_]]}")

println(s"Classe física: ${dfVendas.getClass.getName}")

df é um Dataset[Row]? true
Classe física: org.apache.spark.sql.Dataset


In [18]:
import org.apache.spark.rdd.RDD

val df: DataFrame = Seq(
  (1L, "Ana", "Eletrônicos", 2500.00, "SP"),
  (2L, "Bruno", "Roupas", 150.50, "RJ")
).toDF("id", "cliente", "categoria", "valor", "estado")

// 1. Convertendo explicitamente e atribuindo o tipo RDD[Row]
val rddLinhas: RDD[Row] = df.rdd 
println(s"Tipo do atributo .rdd: ${rddLinhas.getClass.getSimpleName}")

Tipo do atributo .rdd: MapPartitionsRDD


import org.apache.spark.rdd.RDD
df: DataFrame = [id: bigint, cliente: string ... 3 more fields]
rddLinhas: RDD[Row] = MapPartitionsRDD[52] at rdd at cmd18.sc:9

- Execução de transformação de dados

Lista de funções suportadas pelo SparkSQL: https://spark.apache.org/docs/latest/api/java/org/apache/spark/sql/functions.html

In [19]:
println("=== operações via select, filter, withColumn, groupBy ===")

val dfOtimizadoDSL = dfVendas
  .select($"cliente", $"categoria", $"valor", $"estado")
  .filter($"estado" === "SP" && $"valor" >= 1000.00)
  .withColumn("imposto_10pct", $"valor" * 0.10)
  .withColumn("valor_total", $"valor" + $"imposto_10pct")

dfOtimizadoDSL.show()

=== operações via select, filter, withColumn, groupBy ===
+--------------+----------------+------+------+------------------+------------------+
|       cliente|       categoria| valor|estado|     imposto_10pct|       valor_total|
+--------------+----------------+------+------+------------------+------------------+
| Isabela Pires|       Alimentos|1245.1|    SP|124.50999999999999|           1369.61|
|   Diego Alves|     Eletrônicos|2890.0|    SP|             289.0|            3179.0|
| Gabriela Lima|     Eletrônicos|1599.0|    SP|             159.9|            1758.9|
|   Bruno Souza|Casa e Decoração|1420.0|    SP|             142.0|            1562.0|
|  Hugo Martins|     Eletrônicos|2990.0|    SP|             299.0|            3289.0|
|     Ana Silva|     Eletrônicos|3890.0|    SP|             389.0|            4279.0|
| Isabela Pires|     Eletrônicos|1999.9|    SP|            199.99|2199.8900000000003|
|    Carla Dias|     Eletrônicos|4600.0|    SP|             460.0|            5060

dfOtimizadoDSL: DataFrame = [cliente: string, categoria: string ... 4 more fields]

In [20]:
import org.apache.spark.sql.functions.col

val dfDSLCol = dfVendas
  .select(col("cliente"), col("estado"))
  .filter(col("estado") === "SP")

dfDSLCol.show()

+-------------+------+
|      cliente|estado|
+-------------+------+
|Isabela Pires|    SP|
|  Diego Alves|    SP|
|    Ana Silva|    SP|
|Gabriela Lima|    SP|
|  Bruno Souza|    SP|
| Hugo Martins|    SP|
|  Bruno Souza|    SP|
| Hugo Martins|    SP|
|    Ana Silva|    SP|
|Isabela Pires|    SP|
|   Carla Dias|    SP|
|Isabela Pires|    SP|
|   Carla Dias|    SP|
|Isabela Pires|    SP|
|  Diego Alves|    SP|
|   João Pedro|    SP|
|   Carla Dias|    SP|
|Isabela Pires|    SP|
|  Diego Alves|    SP|
|   João Pedro|    SP|
+-------------+------+
only showing top 20 rows



import org.apache.spark.sql.functions.col
dfDSLCol: Dataset[Row] = [cliente: string, estado: string]

In [21]:
import org.apache.spark.sql.functions._

// Agrupamento e Agregação
val dfAgrupadoDSL = dfVendas
  .groupBy($"categoria")
  .agg(
    sum($"valor").as("faturamento_total"),
    avg($"valor").as("ticket_medio"),
    count($"id_venda").as("qtd_vendas")
  )
  .orderBy($"faturamento_total".desc)

dfAgrupadoDSL.show()

+----------------+------------------+------------------+----------+
|       categoria| faturamento_total|      ticket_medio|qtd_vendas|
+----------------+------------------+------------------+----------+
|     Eletrônicos|123711.70000000001|3172.0948717948722|        39|
|Casa e Decoração|          33890.39| 996.7761764705882|        34|
|       Alimentos|            9305.3|265.86571428571426|        35|
|          Roupas|           8291.05|       259.0953125|        32|
|          Livros|           7107.52|            253.84|        28|
|         Higiene|            1761.1|         55.034375|        32|
+----------------+------------------+------------------+----------+



import org.apache.spark.sql.functions._
dfAgrupadoDSL: Dataset[Row] = [categoria: string, faturamento_total: double ... 2 more fields]

- Execução de consultas SQL

Também é possível executar consultas SQL de forma direta, para realizar isso nós geralmente cadastramos um dataframe como se fosse uma tabela ou View no catálogo do SparkSession (https://spark.apache.org/docs/latest/api/scala/org/apache/spark/sql/catalog/Catalog.html).

As consultas SQL seguem o SQL ANSI (https://spark.apache.org/docs/latest/sql-ref-ansi-compliance.html) permitindo que funções e operadores do SQL padrão possam ser utilizados: SELECT, WHERE, GROUPBY, ORDER BY, JOIN, EXISTS, etc. 

In [22]:
// Registra o DataFrame como uma tabela temporária no catálogo da SparkSession
dfVendas.createOrReplaceTempView("tb_vendas")

val dfResultadoSQL = spark.sql("""
  SELECT 
    categoria,
    SUM(valor) AS faturamento_total,
    AVG(valor) AS ticket_medio,
    COUNT(id_venda) AS qtd_vendas
  FROM tb_vendas
  WHERE estado IN ('SP', 'RJ')
  GROUP BY categoria
  HAVING SUM(valor) > 500
  ORDER BY faturamento_total DESC
""")

dfResultadoSQL.show()


+----------------+-----------------+------------------+----------+
|       categoria|faturamento_total|      ticket_medio|qtd_vendas|
+----------------+-----------------+------------------+----------+
|     Eletrônicos|          55908.8|3106.0444444444447|        18|
|Casa e Decoração|         10629.99|          885.8325|        12|
|       Alimentos|           3170.3|288.20909090909095|        11|
|          Roupas|          3065.15|           306.515|        10|
|          Livros|            550.0|             110.0|         5|
+----------------+-----------------+------------------+----------+



dfResultadoSQL: DataFrame = [categoria: string, faturamento_total: double ... 2 more fields]

In [23]:
// O .explain() exibe o plano de execução gerado pelo Catalyst
dfResultadoSQL.explain(extended = true)

val df_temp = dfResultadoSQL

== Parsed Logical Plan ==
'Sort ['faturamento_total DESC NULLS LAST], true
+- 'UnresolvedHaving ('SUM('valor) > 500)
   +- 'Aggregate ['categoria], ['categoria, 'SUM('valor) AS faturamento_total#246, 'AVG('valor) AS ticket_medio#247, 'COUNT('id_venda) AS qtd_vendas#248]
      +- 'Filter 'estado IN (SP,RJ)
         +- 'UnresolvedRelation [tb_vendas], [], false

== Analyzed Logical Plan ==
categoria: string, faturamento_total: double, ticket_medio: double, qtd_vendas: bigint
Sort [faturamento_total#246 DESC NULLS LAST], true
+- Filter (faturamento_total#246 > cast(500 as double))
   +- Aggregate [categoria#19], [categoria#19, sum(valor#20) AS faturamento_total#246, avg(valor#20) AS ticket_medio#247, count(id_venda#17) AS qtd_vendas#248L]
      +- Filter estado#21 IN (SP,RJ)
         +- SubqueryAlias tb_vendas
            +- View (`tb_vendas`, [id_venda#17,cliente#18,categoria#19,valor#20,estado#21,data_venda#22])
               +- Relation [id_venda#17,cliente#18,categoria#19,valor#20,es

df_temp: DataFrame = [categoria: string, faturamento_total: double ... 2 more fields]

In [24]:
df_temp.collect()

println("\n=== APÓS A EXECUÇÃO (Plano Real / Otimizado) ===")
// Exibirá: AdaptiveSparkPlan isFinalPlan=true
df_temp.explain()


=== APÓS A EXECUÇÃO (Plano Real / Otimizado) ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   *(3) Sort [faturamento_total#246 DESC NULLS LAST], true, 0
   +- AQEShuffleRead coalesced
      +- ShuffleQueryStage 1
         +- Exchange rangepartitioning(faturamento_total#246 DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=296]
            +- *(2) Filter (isnotnull(faturamento_total#246) AND (faturamento_total#246 > 500.0))
               +- *(2) HashAggregate(keys=[categoria#19], functions=[sum(valor#20), avg(valor#20), count(id_venda#17)])
                  +- AQEShuffleRead coalesced
                     +- ShuffleQueryStage 0
                        +- Exchange hashpartitioning(categoria#19, 200), ENSURE_REQUIREMENTS, [plan_id=257]
                           +- *(1) HashAggregate(keys=[categoria#19], functions=[partial_sum(valor#20), partial_avg(valor#20), partial_count(id_venda#17)])
                              +- *(1) Project [id_venda#17, ca

res24_0: Array[Row] = Array(
  [Eletrônicos,55908.8,3106.0444444444447,18],
  [Casa e Decoração,10629.99,885.8325,12],
  [Alimentos,3170.3,288.20909090909095,11],
  [Roupas,3065.15,306.515,10],
  [Livros,550.0,110.0,5]
)

 # Tipos de Join no SparkSQL

[ INNER ] | CROSS | LEFT [ OUTER ] | [ LEFT ] SEMI | RIGHT [ OUTER ] | FULL [ OUTER ] | [ LEFT ] ANTI

https://spark.apache.org/docs/latest/sql-ref-syntax-qry-select-join.html

In [26]:
println("=== Join ===")

// DataFrame menor de metadados das categorias (Tabela Dimensão)
val dfCategoriasInfo = Seq(
  ("Eletrônicos", "Setor A", "Garantia Estendida"),
  ("Roupas", "Setor B", "Troca 30 dias"),
  ("Higiene", "Setor C", "Sem Troca")
).toDF("categoria", "setor", "politica")

val dfVendasComMeta = dfVendas.join(
  dfCategoriasInfo,
  Seq("categoria"), // Coluna chave de junção
  "inner"
)

dfVendasComMeta.show()

println("=== Plano Físico de Execução (Catalyst) ===")

// O .explain() exibe o plano de execução gerado pelo Catalyst
dfVendasComMeta.explain(extended = true)

=== Join ===
+-----------+--------+--------------+------+------+----------+-------+------------------+
|  categoria|id_venda|       cliente| valor|estado|data_venda|  setor|          politica|
+-----------+--------+--------------+------+------+----------+-------+------------------+
|Eletrônicos|       3|     Ana Silva|4812.9|    PR|2024-07-22|Setor A|Garantia Estendida|
|     Roupas|       4|  Hugo Martins|890.15|    RJ|2024-01-10|Setor B|     Troca 30 dias|
|    Higiene|       6|   Bruno Souza|  45.9|    BA|2024-05-18|Setor C|         Sem Troca|
|Eletrônicos|       7|   Diego Alves|2890.0|    SP|2024-02-11|Setor A|Garantia Estendida|
|     Roupas|       8|   Elena Costa| 120.5|    MG|2024-08-30|Setor B|     Troca 30 dias|
|Eletrônicos|      12|   Bruno Souza|4120.0|    PR|2024-10-19|Setor A|Garantia Estendida|
|    Higiene|      13|    Carla Dias|  32.5|    RJ|2024-01-25|Setor C|         Sem Troca|
|     Roupas|      16|Fernando Rocha| 240.0|    MG|2024-09-08|Setor B|     Troca 30 dia

dfCategoriasInfo: DataFrame = [categoria: string, setor: string ... 1 more field]
dfVendasComMeta: DataFrame = [categoria: string, id_venda: int ... 6 more fields]

# Escrita de dados no Spark

In [28]:
// Grava os dados em múltiplos arquivos CSV com cabeçalho,
// criando subpastas físicas segregadas por 'estado' (ex: estado=SP/, estado=RJ/)
dfVendas.write
  .mode("overwrite")
  .option("header", "true")
  .partitionBy("estado")
  .csv("saida_vendas_particionada_csv")

println("Dados gravados com sucesso no diretório 'saida_vendas_particionada.csv'!")

Dados gravados com sucesso no diretório 'saida_vendas_particionada.csv'!


# Particionamento de Dados sobre os DataFrames 

In [28]:
import $ivy.`org.apache.spark::spark-sql:3.5.0`


import org.apache.spark.sql.SparkSession
import org.apache.spark.sql.functions._

val spark = SparkSession.builder()
  .appName("ParticionamentoEShuffle")
  .master("local[2]") // Utiliza todas as threads da CPU local
  .getOrCreate()

import spark.implicits._

val dfVendas = Seq(
  (1, "Ana", "SP", 2500.0),
  (2, "Bruno", "RJ", 150.0),
  (3, "Carla", "SP", 3200.0),
  (4, "Diego", "MG", 45.0),
  (5, "Elena", "SP", 220.0)
).toDF("id", "cliente", "estado", "valor")


// O padrão em memória depende dos núcleos da CPU (ex: local[*] = número de threads)
println(s"1. Partições padrões em memória: ${dfVendas.rdd.getNumPartitions}")

// spark.sql.shuffle.partitions)
// Operações como groupBy ou join redistribuem os dados entre os nós. Por padrão no Spark SQL, essa redistribuição cria 200 partições.
val dfAgrupado = dfVendas.groupBy($"estado").sum("valor")

println(s"2. Partições após Shuffle (padrão do Spark SQL): ${dfAgrupado.rdd.getNumPartitions}")

// Ajustando a propriedade global para evitar 200 partições em datasets pequenos:
spark.conf.set("spark.sql.shuffle.partitions", "4")

// Reparticione em memória usando uma coluna como chave (Partition by Key)
val dfReparticionado = dfVendas.repartition($"estado")

println(s"3. Partições após repartition por 'estado': ${dfReparticionado.rdd.getNumPartitions}")


Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar
Failed to download https://repo1.maven.org/maven2/io/netty/netty-all/4.1.96.Final/netty-all-4.1.96.Final-sources.jar.sha1
Failed to download https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-client-runtime/3.3.4/hadoop-client-runtime-3.3.4-sources.jar.sha1
cmd29.sc:20: value toDF is not a member of Seq[(Int, String, String, Double)]
possible cause: maybe a semicolon is missing before `value toDF`?
).toDF("id", "cliente", "estado", "valor")
  ^
Compilation Failed